In [2]:
import pandas as pd
import polars as pl
import numpy as np
import time
import os
import json
import random

# --- Configuration for generating large JSON data ---
num_records = 100000 # Number of records in the large JSON file
json_file_name = 'large_data.json' # Name of the generated JSON file

print("--- Generating Large JSON Data ---")
print(f"Creating a JSON file with {num_records:,} records: '{json_file_name}'")

# Generate synthetic JSON data
def generate_large_json_data(num_records):
    data = []
    cities = [f"city_{i}" for i in range(20)]
    categories = [f"cat_{i}" for i in range(50)]
    for i in range(num_records):
        record = {
            "id": i,
            "name": f"Person_{i}",
            "age": random.randint(18, 70),
            "city": random.choice(cities),
            "email": f"person_{i}@example.com",
            "value": random.randint(1, 1000), # Value for filtering and aggregation
            "category": random.choice(categories), # Category for grouping
            "orders": [
                {"order_id": f"ORD{i}_{j}", "amount": round(random.uniform(10, 500), 2)}
                for j in range(random.randint(0, 3)) # 0 to 3 orders per person
            ]
        }
        data.append(record)
    return data

# start_time_gen = time.time()
# large_json_data = generate_large_json_data(num_records)
# with open(json_file_name, 'w') as f:
#     json.dump(large_json_data, f)
# end_time_gen = time.time()
# print(f"Large JSON data generation completed in: {end_time_gen - start_time_gen:.4f} seconds")
# print("-" * 50)
# --- Generating Large JSON Data ---
# Creating a JSON file with 100,000 records: 'large_data.json'
# Large JSON data generation completed in: 10.5922 seconds
# --------------------------------------------------

--- Generating Large JSON Data ---
Creating a JSON file with 100,000 records: 'large_data.json'


In [3]:
print("\n--- Pandas Execution ---")
start_time_pandas = time.time()

json_file_name='large_data.json'
# 1. Read JSON data
# Pandas can directly read JSON files. For large files, `lines=True` might be needed
# if each line is a separate JSON object (JSONL format), but here it's an array of objects.
df_pd = pd.read_json(json_file_name)
print(f"Pandas DataFrame shape: {df_pd.shape}")
print("Pandas DataFrame head:")
print(df_pd.head())

# 2. Filter data:
# - value > 500
# - category starts with 'cat_5' (adjusting to the new category names)
# - city is 'city_10'
# We'll filter on 'category' starting with 'cat_4' and 'city_1' to ensure some data.
df_pd_filtered = df_pd[
    (df_pd["value"] > 500) &
    (df_pd["category"].str.startswith("cat_4")) &
    (df_pd["city"] == "city_1")]

# 3. Group by and aggregate (mean of 'value' by 'category' and 'city')
df_pd_grouped = df_pd_filtered.groupby(["category", "city"])["value"].mean().reset_index()

# 4. Add a new derived column (e.g., scaled_value)
df_pd_grouped["scaled_value"] = df_pd_grouped["value"] * 1.5

end_time_pandas = time.time()
pandas_execution_time = end_time_pandas - start_time_pandas
print(f"Pandas total execution time: {pandas_execution_time:.4f} seconds")
print("-" * 50)


print("\n--- Polars Execution ---")
start_time_polars = time.time()

# 1. Read JSON data directly into Polars DataFrame
# Polars has optimized readers for various file formats, including JSON.
# `pl.read_json()` can directly handle JSON arrays of objects.
df_pl = pl.read_json(json_file_name)
print(f"Polars DataFrame shape: {df_pl.shape}")
print("Polars DataFrame head:")
print(df_pl.head())

# Convert to LazyFrame to leverage Polars' optimization for chained operations
df_pl_lazy = df_pl.lazy()

# 2. Filter data (operations on LazyFrame build the query plan)
df_pl_filtered_lazy = df_pl_lazy.filter(
    (pl.col("value") > 500) &
    (pl.col("category").str.starts_with("cat_4")) &
    (pl.col("city") == "city_1"))

# 3. Group by and aggregate
df_pl_grouped_lazy = df_pl_filtered_lazy.group_by(["category", "city"]).agg(
    pl.col("value").mean().alias("value"))

# 4. Add a new derived column and trigger computation with .collect()
df_pl_final = df_pl_grouped_lazy.with_columns(
    (pl.col("value") * 1.5).alias("scaled_value")).collect() # .collect() triggers the actual execution

print(f"Polars final shape: {df_pl_final.shape}")
print("Polars final DataFrame head:")
print(df_pl_final.head())

end_time_polars = time.time()
polars_execution_time = end_time_polars - start_time_polars
print(f"Polars total execution time: {polars_execution_time:.4f} seconds")
print("-" * 50)


# --- Comparison ---
print("\n--- Execution Time Comparison ---")
print(f"Pandas took: {pandas_execution_time:.4f} seconds")
print(f"Polars took: {polars_execution_time:.4f} seconds")

if pandas_execution_time > polars_execution_time:
    speed_up = pandas_execution_time / polars_execution_time
    print(f"Polars was approximately {speed_up:.2f} times faster than Pandas for this task.")
elif polars_execution_time > pandas_execution_time:
    speed_up = polars_execution_time / pandas_execution_time
    print(f"Pandas was approximately {speed_up:.2f} times faster than Polars for this task (less common for large datasets and complex operations).")
else:
    print("Execution times were similar.")

# --- Cleanup ---
# os.remove(json_file_name) # Uncomment to remove the generated file after execution
# print(f"\nCleaned up: '{json_file_name}' removed.")



--- Pandas Execution ---
Pandas DataFrame shape: (100000, 8)
Pandas DataFrame head:
   id      name  age     city                 email  value category  \
0   0  Person_0   62   city_5  person_0@example.com    914   cat_43   
1   1  Person_1   54   city_4  person_1@example.com    423   cat_22   
2   2  Person_2   22  city_13  person_2@example.com    878   cat_34   
3   3  Person_3   45  city_18  person_3@example.com    892   cat_29   
4   4  Person_4   34   city_7  person_4@example.com    410   cat_38   

                                              orders  
0  [{'order_id': 'ORD0_0', 'amount': 296.02}, {'o...  
1                                                 []  
2                                                 []  
3                                                 []  
4  [{'order_id': 'ORD4_0', 'amount': 29.04}, {'or...  
Pandas total execution time: 2.3609 seconds
--------------------------------------------------

--- Polars Execution ---
Polars DataFrame shape: (100000, 8)
P

In [4]:
# # --- Configuration for generating large JSON data ---
# num_records = 100000 # Number of records in the large JSON file
# json_file_name = 'large_nested_data.json' # Name of the generated JSON file

# print("--- Generating Large Nested JSON Data ---")
# print(f"Creating a JSON file with {num_records:,} records, nested under 'data' key: '{json_file_name}'")

# # Generate synthetic JSON data with a nested structure
# def generate_large_nested_json_data(num_records):
#     records = []
#     cities = [f"city_{i}" for i in range(20)]
#     categories = [f"cat_{i}" for i in range(50)]
#     for i in range(num_records):
#         record = {
#             "id": i,
#             "name": f"Person_{i}",
#             "age": random.randint(18, 70),
#             "location": { # Nested object for location
#                 "city": random.choice(cities),
#                 "country": "USA"
#             },
#             "email": f"person_{i}@example.com",
#             "value": random.randint(1, 1000), # Value for filtering and aggregation
#             "category": random.choice(categories), # Category for grouping
#             "orders": [
#                 {"order_id": f"ORD{i}_{j}", "amount": round(random.uniform(10, 500), 2)}
#                 for j in range(random.randint(0, 3)) # 0 to 3 orders per person
#             ]
#         }
#         records.append(record)

#     # Wrap the list of records in a dictionary under a 'data' key
#     nested_data = {
#         "metadata": {
#             "source": "synthetic_generator",
#             "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
#             "total_records": num_records
#         },
#         "data": records
#     }
#     return nested_data

# # start_time_gen = time.time()
# # large_nested_json_data = generate_large_nested_json_data(num_records)
# # with open(json_file_name, 'w') as f:
# #     json.dump(large_nested_json_data, f, indent=None) # No indent for larger files
# # end_time_gen = time.time()
# # print(f"Large nested JSON data generation completed in: {end_time_gen - start_time_gen:.4f} seconds")
# # print("-" * 50)
# # --- Generating Large Nested JSON Data ---
# # Creating a JSON file with 100,000 records, nested under 'data' key: 'large_nested_data.json'
# # Large nested JSON data generation completed in: 8.6392 seconds
# # --------------------------------------------------

--- Generating Large Nested JSON Data ---
Creating a JSON file with 100,000 records, nested under 'data' key: 'large_nested_data.json'
Large nested JSON data generation completed in: 8.6392 seconds
--------------------------------------------------


In [5]:
print("\n--- Pandas Execution ---")
start_time_pandas = time.time()

json_file_name='large_nested_data.json'
# 1. Read JSON data
# For nested JSON, we first load the entire JSON into a Python dictionary,
# then extract the list of records from the 'data' key.
with open(json_file_name, 'r') as f:
    raw_json_data_pandas = json.load(f)
data_for_df_pandas = raw_json_data_pandas['data']

# Now create DataFrame from the list of records
df_pd = pd.DataFrame(data_for_df_pandas)

# Pandas can attempt to normalize nested dictionaries like 'location'
# For 'location', it automatically creates a dictionary column. We need to normalize it.
df_pd = pd.json_normalize(data_for_df_pandas, sep='_')


print(f"Pandas DataFrame shape: {df_pd.shape}")
print("Pandas DataFrame head:")
print(df_pd.head())

# 2. Filter data:
# - value > 500
# - category starts with 'cat_4'
# - city is 'city_1' (now accessible via 'location_city')
df_pd_filtered = df_pd[
    (df_pd["value"] > 500) &
    (df_pd["category"].str.startswith("cat_4")) &
    (df_pd["location_city"] == "city_1")] # Note the column name change for nested data

# 3. Group by and aggregate (mean of 'value' by 'category' and 'location_city')
df_pd_grouped = df_pd_filtered.groupby(["category", "location_city"])["value"].mean().reset_index()

# 4. Add a new derived column (e.g., scaled_value)
df_pd_grouped["scaled_value"] = df_pd_grouped["value"] * 1.5

end_time_pandas = time.time()
pandas_execution_time = end_time_pandas - start_time_pandas
print(f"Pandas total execution time: {pandas_execution_time:.4f} seconds")
print("-" * 50)


print("\n--- Polars Execution ---")
start_time_polars = time.time()

# 1. Read JSON data directly into Polars DataFrame
# Polars `read_json` can often handle top-level objects with a specific array key.
# If `read_json` struggles with deeply nested JSON, `json.load` followed by `pl.DataFrame` is an option.
# For this specific nested structure, we'll load the full JSON first and extract the list.
with open(json_file_name, 'r') as f:
    raw_json_data_polars = json.load(f)
data_for_df_polars = raw_json_data_polars['data']

df_pl = pl.DataFrame(data_for_df_polars)

# Polars handles nested structs naturally, so 'location' will be a struct column.
# We can unnest it easily.
df_pl = df_pl.unnest("location")

print(f"Polars DataFrame shape: {df_pl.shape}")
print("Polars DataFrame head:")
print(df_pl.head())

# Convert to LazyFrame to leverage Polars' optimization for chained operations
df_pl_lazy = df_pl.lazy()

# 2. Filter data (operations on LazyFrame build the query plan)
# Access 'city' directly after unnesting 'location'
df_pl_filtered_lazy = df_pl_lazy.filter(
    (pl.col("value") > 500) &
    (pl.col("category").str.starts_with("cat_4")) &
    (pl.col("city") == "city_1")) # Now 'city' is a top-level column after unnesting

# 3. Group by and aggregate
df_pl_grouped_lazy = df_pl_filtered_lazy.group_by(["category", "city"]).agg(
    pl.col("value").mean().alias("value"))

# 4. Add a new derived column and trigger computation with .collect()
df_pl_final = df_pl_grouped_lazy.with_columns(
    (pl.col("value") * 1.5).alias("scaled_value")).collect() # .collect() triggers the actual execution

print(f"Polars final shape: {df_pl_final.shape}")
print("Polars final DataFrame head:")
print(df_pl_final.head())

end_time_polars = time.time()
polars_execution_time = end_time_polars - start_time_polars
print(f"Polars total execution time: {polars_execution_time:.4f} seconds")
print("-" * 50)


# --- Comparison ---
print("\n--- Execution Time Comparison ---")
print(f"Pandas took: {pandas_execution_time:.4f} seconds")
print(f"Polars took: {polars_execution_time:.4f} seconds")

if pandas_execution_time > polars_execution_time:
    speed_up = pandas_execution_time / polars_execution_time
    print(f"Polars was approximately {speed_up:.2f} times faster than Pandas for this task.")
elif polars_execution_time > pandas_execution_time:
    speed_up = polars_execution_time / pandas_execution_time
    print(f"Pandas was approximately {speed_up:.2f} times faster than Polars for this task (less common for large datasets and complex operations).")
else:
    print("Execution times were similar.")

# --- Cleanup ---
# os.remove(json_file_name) # Uncomment to remove the generated file after execution
# print(f"\nCleaned up: '{json_file_name}' removed.")



--- Pandas Execution ---
Pandas DataFrame shape: (100000, 9)
Pandas DataFrame head:
   id      name  age                 email  value category  \
0   0  Person_0   45  person_0@example.com    564   cat_17   
1   1  Person_1   30  person_1@example.com    964   cat_25   
2   2  Person_2   55  person_2@example.com    462    cat_8   
3   3  Person_3   23  person_3@example.com    189   cat_33   
4   4  Person_4   67  person_4@example.com    257    cat_5   

                                              orders location_city  \
0         [{'order_id': 'ORD0_0', 'amount': 447.75}]       city_12   
1                                                 []        city_4   
2  [{'order_id': 'ORD2_0', 'amount': 48.55}, {'or...       city_16   
3                                                 []       city_12   
4         [{'order_id': 'ORD4_0', 'amount': 485.19}]       city_18   

  location_country  
0              USA  
1              USA  
2              USA  
3              USA  
4              U